In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output

# Step 1: Detect Hardware
# CUDA = NVIDIA GPU, MPS = Apple M-series, CPU = Default
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"✅ Running on: {device.type.upper()}")

transform = transforms.Compose([transforms.ToTensor()])

# Step 2: Load Datasets
train_set = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_set = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# Step 3: Create Loaders
# pin_memory=True speeds up the transfer from CPU to GPU
train_loader = DataLoader(train_set, batch_size=64, shuffle=True, pin_memory=True if device.type != 'cpu' else False)
test_loader = DataLoader(test_set, batch_size=64, shuffle=False)

print(f"📦 Data ready for {device.type.upper()} transfer.")

✅ Running on: CUDA
📦 Data ready for CUDA transfer.


In [3]:
model_digits = None

# UI Widgets
neuron_slider = widgets.IntSlider(value=64, min=8, max=256, step=8, description='Complexity:')
epoch_slider = widgets.IntSlider(value=3, min=1, max=10, description='Study Time:')
activation_map = {'relu': nn.ReLU(), 'sigmoid': nn.Sigmoid(), 'tanh': nn.Tanh()}
activation_dropdown = widgets.Dropdown(options=['relu', 'sigmoid', 'tanh'], value='relu', description='Logic Style:')
btn_train = widgets.Button(description="🚀 Train on " + device.type.upper(), button_style='success')
output = widgets.Output()

def train_model(b):
    global model_digits
    
    with output:
        clear_output()
        print(f"🚀 Initializing MNIST training on {device}...")

        # 1. Build & Move Model to GPU
        model_digits = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28*28, neuron_slider.value),
            activation_map[activation_dropdown.value],
            nn.Linear(neuron_slider.value, 10)
        ).to(device)

        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model_digits.parameters(), lr=0.001)

        # 2. Training Loop
        model_digits.train()
        total_epochs = epoch_slider.value
        
        for epoch in range(total_epochs):
            running_loss = 0.0
            for batch_idx, (images, labels) in enumerate(train_loader):
                # 3. MOVE DATA TO GPU
                images, labels = images.to(device), labels.to(device) 
                
                optimizer.zero_grad()
                outputs = model_digits(images)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
                running_loss += loss.item()
            
            # --- Per-Epoch Status Update ---
            avg_loss = running_loss / len(train_loader)
            clear_output(wait=True)
            print(f"🧠 Training MNIST Brain (Neurons: {neuron_slider.value})")
            print(f"Epoch: {epoch+1}/{total_epochs}")
            print(f"📉 Loss: {avg_loss:.4f}")
            # Visual Progress Bar
            progress = int((epoch + 1) / total_epochs * 20)
            print(f"[{'=' * progress}{' ' * (20 - progress)}]")

        print(f"\n🏁 Training Finished on {device.type.upper()}!")

        # 4. Inference (Test on one sample)
        model_digits.eval()
        with torch.no_grad():
            images, labels = next(iter(test_loader))
            test_img, test_lbl = images[0].to(device), labels[0].to(device)
            
            output_pred = model_digits(test_img.unsqueeze(0))
            prediction = torch.argmax(output_pred, dim=1).item()

            # Move back to CPU for plotting
            plt.figure(figsize=(3,3))
            plt.imshow(test_img.cpu().squeeze(), cmap='gray')
            plt.title(f"AI Guess: {prediction} | Actual: {test_lbl.item()}")
            plt.axis('off')
            plt.show()

btn_train.on_click(train_model)
ui = widgets.VBox([widgets.HTML("<b>Hardware-Accelerated Training:</b>"),
                   neuron_slider, activation_dropdown, epoch_slider, btn_train])
display(ui, output)

Output()

In [4]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

# Create a button to generate the report
btn_report = widgets.Button(description="📊 Generate Report Card", button_style='info')
report_output = widgets.Output()

def show_confusion_matrix(b):
    with report_output:
        clear_output()
        
        if model_digits is None:
            print("❌ Please train your AI in Cell 2 first!")
            return

        # 1. Switch model to Evaluation Mode
        model_digits.eval()
        
        # 2. Get a batch of test data
        images, labels = next(iter(test_loader))
        
        # MOVE images to the same device as the model (GPU/MPS)
        images = images.to(device)
        
        # Tell the model to guess without tracking gradients
        with torch.no_grad():
            outputs = model_digits(images)
            
            # 3. MOVE BACK TO CPU for NumPy/Sklearn
            # .cpu() brings the data back to the main processor
            predicted_labels = torch.argmax(outputs, dim=1).cpu().numpy()
            true_labels = labels.numpy() # Original labels are already on CPU

        # 4. Create the Matrix
        cm = confusion_matrix(true_labels, predicted_labels)

        # 5. Plot it beautifully
        plt.figure(figsize=(10, 8))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=range(10), yticklabels=range(10))
        plt.xlabel('What the AI GUESSED')
        plt.ylabel('What the digit ACTUALLY was')
        plt.title(f'The Confusion Matrix (Processed on {device.type.upper()})')
        plt.show()

        print("💡 Pro-Tip: Look for numbers outside the diagonal line.")
        print("If you see a high number at Row 4, Column 9, your AI thinks 4s look like 9s!")

btn_report.on_click(show_confusion_matrix)
display(btn_report, report_output)

Button(button_style='info', description='📊 Generate Report Card', style=ButtonStyle())

Output()

In [5]:
btn_visualize = widgets.Button(description="👁️ See AI Features", button_style='warning')
vis_output = widgets.Output()

def visualize_weights(b):
    with vis_output:
        clear_output()
        if model_digits is None:
            print("❌ Error: Train the AI first!")
            return

        # 1. Identify the first linear layer
        # In our Sequential: [Flatten, Linear, Activation, Linear]
        # index 1 is our first set of weights
        first_layer = model_digits[1]
        
        # 2. Extract weights and MOVE TO CPU
        # .cpu() ensures we can convert to a NumPy array for plotting
        weights = first_layer.weight.detach().cpu().numpy()

        # 3. Determine how many neurons to show
        num_neurons = min(10, weights.shape[0])
        plt.figure(figsize=(15, 3))

        for i in range(num_neurons):
            # 4. Reshape weights back into a 28x28 square
            neuron_weights = weights[i, :].reshape(28, 28)

            plt.subplot(1, num_neurons, i + 1)
            # RdBu map: Red = "I like pixels here", Blue = "I hate pixels here"
            plt.imshow(neuron_weights, cmap='RdBu', vmin=-0.15, vmax=0.15)
            plt.title(f"Neuron {i+1}")
            plt.axis('off')

        plt.suptitle("AI Feature Stencils: What the Neurons See (Red=Positive, Blue=Negative)", fontsize=14)
        plt.show()
        print(f"💡 Visualizing weights currently stored on {device.type.upper()}")
        print("Each square is a filter looking for specific curves or edges!")

btn_visualize.on_click(visualize_weights)
display(btn_visualize, vis_output)

Button(button_style='warning', description='👁️ See AI Features', style=ButtonStyle())

Output()

In [8]:
import torch
import torch.nn.functional as F
import numpy as np
import base64
import io
from PIL import Image
from IPython.display import HTML, display, clear_output
import ipywidgets as widgets
import matplotlib.pyplot as plt

# 1. Prediction logic (Device-Aware for GPU/MPS)
def on_data_received(change):
    image_data = change['new']
    # Prevent empty triggers
    if not image_data or len(image_data) < 100: return 
    
    try:
        # Decode and pre-process
        binary = base64.b64decode(image_data.split(',')[1])
        img = PILImage.open(io.BytesIO(binary)).convert('L').resize((28, 28))
        img_array = np.array(img) / 255.0
        
        # 🟢 MOVE TO GPU/MPS
        img_tensor = torch.FloatTensor(img_array).unsqueeze(0).unsqueeze(0).to(device)
        
        model_digits.eval()
        with torch.no_grad():
            logits = model_digits(img_tensor)
            probs = torch.nn.functional.softmax(logits, dim=1)
            conf, pred = torch.max(probs, 1)
        
        with output:
            clear_output(wait=True)
            print(f"🤖 AI Result (on {device.type.upper()}):")
            print(f"I am {conf.item()*100:.1f}% sure that is a {pred.item()}!")
            
            plt.imshow(img_array, cmap='gray')
            plt.axis('off')
            plt.show()
            
    except Exception as e:
        with output: print(f"Error: {e}")

# 2. Setup the Hidden Bridge (We give it a unique CSS class)
bridge_widget = widgets.Text(value="")
bridge_widget.add_class("gpu-bridge-input") # This helps JS find it
bridge_widget.observe(on_data_received, names='value')

# 3. UI Display (Updated JavaScript to fix the button)
# I kept your HTML/CSS exactly the same, just fixed the "Identify" click logic
canvas_html = """
<div style="background: #f0f0f0; padding: 20px; border-radius: 15px; width: 320px; border: 3px solid #333; text-align: center;">
    <canvas id="final_canvas" width="280" height="280" style="background: black; cursor: crosshair; border: 2px solid #000; border-radius: 5px;"></canvas>
    <div style="margin-top: 15px; display: flex; justify-content: space-around;">
        <button id="js_predict_btn" style="padding: 10px 15px; background: #28a745; color: white; border: none; border-radius: 5px; cursor: pointer; font-weight: bold;">🔍 Identify</button>
        <button id="js_clear_btn" style="padding: 10px 15px; background: #dc3545; color: white; border: none; border-radius: 5px; cursor: pointer; font-weight: bold;">🧹 Clear</button>
    </div>
</div>

<script>
    (function() {
        var canvas = document.getElementById('final_canvas');
        var ctx = canvas.getContext('2d');
        ctx.strokeStyle = "white"; ctx.lineWidth = 22; ctx.lineCap = "round";
        var drawing = false;

        canvas.onmousedown = () => { drawing = true; };
        canvas.onmouseup = () => { drawing = false; ctx.beginPath(); };
        canvas.onmousemove = (e) => {
            if (!drawing) return;
            var rect = canvas.getBoundingClientRect();
            ctx.lineTo(e.clientX - rect.left, e.clientY - rect.top);
            ctx.stroke(); ctx.beginPath(); ctx.moveTo(e.clientX - rect.left, e.clientY - rect.top);
        };

        document.getElementById('js_clear_btn').onclick = function() {
            ctx.clearRect(0, 0, canvas.width, canvas.height);
        };

        document.getElementById('js_predict_btn').onclick = function() {
            var dataURL = canvas.toDataURL('image/png');
            // FIX: Find the most recent bridge widget by its class
            var bridgeInputs = document.querySelectorAll('.gpu-bridge-input input');
            if (bridgeInputs.length > 0) {
                var bridge = bridgeInputs[bridgeInputs.length - 1]; 
                bridge.value = dataURL;
                bridge.dispatchEvent(new Event('input', { bubbles: true }));
            } else {
                alert("Python bridge not found! Please re-run the cell.");
            }
        };
    })();
</script>
"""

output = widgets.Output()
bridge_ui = widgets.Box([bridge_widget], layout=widgets.Layout(display='none'))

display(HTML(canvas_html))
display(bridge_ui, output)

Box(children=(Text(value='', _dom_classes=('gpu-bridge-input',)),), layout=Layout(display='none'))

Output()

In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import base64
import io
from PIL import Image as PILImage

# 1. Hardware Detection
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"✅ Showdown hardware: {device.type.upper()}")

# 2. Setup Data Transformations
# Note: EMNIST is "transposed" by default. We flip it to match human drawing.
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
    transforms.Lambda(lambda x: x.transpose(1, 2)) 
])

# 3. Load EMNIST Letters
train_data = torchvision.datasets.EMNIST(root='./data', split='letters', train=True, download=True, transform=transform)
test_data = torchvision.datasets.EMNIST(root='./data', split='letters', train=False, download=True, transform=transform)

# 4. Create DataLoader Scenarios
# pin_memory=True speeds up the "handoff" to the GPU
kwargs = {'pin_memory': True, 'num_workers': 0} if device.type != 'cpu' else {}

# Narrow: Simulating biased, limited data
narrow_loader = DataLoader(Subset(train_data, range(500)), batch_size=64, shuffle=True, **kwargs)

# Diverse: Simulating robust, varied data
diverse_loader = DataLoader(Subset(train_data, range(10000)), batch_size=64, shuffle=True, **kwargs)

# Global Test Loader
test_loader = DataLoader(test_data, batch_size=1000, shuffle=True, **kwargs)

alphabet = " ABCDEFGHIJKLMNOPQRSTUVWXYZ"
print(f"✅ Data Ready! Narrow set: 500 samples | Diverse set: 10,000 samples.")

✅ Showdown hardware: CUDA
✅ Data Ready! Narrow set: 500 samples | Diverse set: 10,000 samples.


In [22]:
import time
import torch
import torch.nn as nn
import torch.optim as optim
import ipywidgets as widgets
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
from IPython.display import display, clear_output

# 1. THE ARCHITECTURES
class EMNIST_Fixer(nn.Module):
    def forward(self, x):
        # Fixes the EMNIST transposition (swaps W and H)
        return x.transpose(-1, -2)

class NarrowNet(nn.Module):
    def __init__(self):
        super(NarrowNet, self).__init__()
        self.main = nn.Sequential(
            EMNIST_Fixer(),
            nn.Flatten(),
            nn.Linear(28*28, 128),
            nn.ReLU(),
            nn.Linear(128, 27)
        )
    def forward(self, x): return self.main(x)

class ProNet(nn.Module):
    def __init__(self, filter_size=32):
        super(ProNet, self).__init__()
        self.features = nn.Sequential(
            EMNIST_Fixer(),
            # Layer 1: Edge detection
            nn.Conv2d(1, filter_size, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            # Layer 2: Shape detection
            nn.Conv2d(filter_size, filter_size * 2, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Flatten(),
            nn.Linear((filter_size * 2) * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 27)
        )
    def forward(self, x): return self.features(x)

# 2. EVALUATION LOGIC (Side-by-Side Heatmaps)
def plot_dual_performance(model_n, model_p, loader):
    model_n.eval()
    model_p.eval()
    res = {'n_preds': [], 'p_preds': [], 'labels': []}
    
    with torch.no_grad():
        for imgs, lbls in loader:
            imgs = imgs.to(device)
            res['labels'].extend((lbls - 1).numpy())
            res['n_preds'].extend(torch.argmax(model_n(imgs), 1).cpu().numpy())
            res['p_preds'].extend(torch.argmax(model_p(imgs), 1).cpu().numpy())

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))
    chars = list("ABCDEFGHIJKLMNOPQRSTUVWXYZ")

    sns.heatmap(confusion_matrix(res['labels'], res['n_preds']), ax=ax1, cmap='rocket_r', xticklabels=chars, yticklabels=chars)
    ax1.set_title("❌ Narrow AI (Biased Data) Error Map")
    
    sns.heatmap(confusion_matrix(res['labels'], res['p_preds']), ax=ax2, cmap='rocket_r', xticklabels=chars, yticklabels=chars)
    ax2.set_title(f"✅ Pro AI ({filter_slider.value} Capacity) Error Map")
    
    plt.tight_layout()
    plt.show()

# 3. UI ELEMENTS
filter_slider = widgets.IntSlider(value=32, min=8, max=64, step=8, description='Pro Capacity:')
epoch_slider = widgets.IntSlider(value=10, min=1, max=50, step=1, description='Epochs:')
btn_train_showdown = widgets.Button(description="🔥 Start GPU Showdown", button_style='danger', layout={'width': '250px'})
showdown_status = widgets.Output()

# 4. TRAINING LOGIC
def train_showdown(b):
    global model_narrow, model_pro # Export for use in the drawing pad
    with showdown_status:
        clear_output()
        
        # Initialize
        model_narrow = NarrowNet().to(device)
        model_pro = ProNet(filter_size=filter_slider.value).to(device)
        criterion = nn.CrossEntropyLoss()
        opt_narrow = optim.Adam(model_narrow.parameters(), lr=0.001)
        opt_pro = optim.Adam(model_pro.parameters(), lr=0.001)
        
        total_epochs = epoch_slider.value
        start_time = time.time()

        try:
            # --- Train Narrow ---
            model_narrow.train()
            for epoch in range(total_epochs):
                rloss = 0.0
                for imgs, lbls in narrow_loader:
                    imgs, lbls = imgs.to(device), lbls.to(device)
                    lbls = lbls - 1
                    opt_narrow.zero_grad()
                    loss = criterion(model_narrow(imgs), lbls)
                    loss.backward(); opt_narrow.step()
                    rloss += loss.item()
                
                clear_output(wait=True)
                print(f"🧠 [1/2] Training Narrow AI | Epoch: {epoch+1}/{total_epochs} | Loss: {rloss/len(narrow_loader):.4f}")
                print(f"[{'=' * int((epoch+1)/total_epochs*20):<20}]")

            # --- Train Pro ---
            model_pro.train()
            for epoch in range(total_epochs):
                rloss = 0.0
                for imgs, lbls in diverse_loader:
                    imgs, lbls = imgs.to(device), lbls.to(device)
                    lbls = lbls - 1
                    opt_pro.zero_grad()
                    loss = criterion(model_pro(imgs), lbls)
                    loss.backward(); opt_pro.step()
                    rloss += loss.item()
                
                clear_output(wait=True)
                print(f"✅ Narrow AI: Ready")
                print(f"🌟 [2/2] Training Pro AI | Epoch: {epoch+1}/{total_epochs} | Loss: {rloss/len(diverse_loader):.4f}")
                print(f"[{'=' * int((epoch+1)/total_epochs*20):<20}]")

            print(f"\n🏆 Training Complete! ({time.time() - start_time:.1f}s)")
            print("📊 Generating Error Maps...")
            plot_dual_performance(model_narrow, model_pro, test_loader)

        except Exception as e:
            print(f"⚠️ Error: {str(e)}")

btn_train_showdown.on_click(train_showdown)
ui = widgets.VBox([widgets.HTML("<b>AI Showdown Configuration:</b>"), filter_slider, epoch_slider, btn_train_showdown])
display(ui, showdown_status)

Output()

In [23]:
# 1. THE BRAIN: Python function that runs BOTH models with Confidence on GPU
def on_showdown_received(change):
    image_data = change['new']
    if not image_data or len(image_data) < 100: return
    
    try:
        binary = base64.b64decode(image_data.split(',')[1])
        img = Image.open(io.BytesIO(binary)).convert('L').resize((28, 28))
        
        # 🟢 FIX: Remove the .T here. Let the model handle the orientation.
        img_array = np.array(img) / 255.0
        img_t = torch.tensor(img_array).float() 
        img_t = (img_t - 0.5) / 0.5 
        
        input_t = img_t.unsqueeze(0).unsqueeze(0).to(device) 

        model_narrow.eval(); model_pro.eval()
        with torch.no_grad():
            logits_n = model_narrow(input_t)
            logits_p = model_pro(input_t)
            
            # Get predictions (EMNIST is 1-indexed for letters)
            # Subtract 1 in training, so here we just map directly
            res_n = alphabet[torch.argmax(logits_n, 1).item() + 1]
            res_p = alphabet[torch.argmax(logits_p, 1).item() + 1]
            
            # Confidence
            conf_n = torch.max(torch.softmax(logits_n, 1)).item() * 100
            conf_p = torch.max(torch.softmax(logits_p, 1)).item() * 100

        with showdown_out:
            clear_output(wait=True)
            print(f"📊 SHOWDOWN RESULTS")
            print(f"Narrow AI: {res_n} ({conf_n:.1f}%)")
            print(f"Pro AI:    {res_p} ({conf_p:.1f}%)")
            
            # Show what the AI is actually seeing to confirm it's not flipped
            plt.imshow(img_array, cmap='gray')
            plt.title("AI Visual Input")
            plt.axis('off')
            plt.show()
            
    except Exception as e:
        with showdown_out: print(f"Error: {e}")

            
# 2. THE BRIDGE: Unique class added to prevent button failure
showdown_bridge = widgets.Text(value="")
showdown_bridge.add_class("sd-bridge-input")
showdown_bridge.observe(on_showdown_received, names='value')

# 3. THE INTERFACE: HTML remains the same, JS updated to find .sd-bridge-input
showdown_html = """
<div style="border: 3px solid #333; width: 320px; text-align: center; padding: 15px; background: #fdfdfd; border-radius: 10px;">
    <p style="margin-top:0; font-family:sans-serif;"><b>Draw and Compare Confidence</b></p>
    <canvas id="sd_canvas" width="280" height="280" style="background:black; border: 2px solid #000; cursor: crosshair;"></canvas>
    <br>
    <button id="sd_predict" style="margin-top:10px; padding:10px; background:#28a745; color:white; border:none; border-radius:5px; cursor:pointer; width: 100%; font-weight:bold;">🔮 Compare Models</button>
    <button id="sd_clear" style="margin-top:5px; padding:5px; background:#666; color:white; border:none; border-radius:5px; cursor:pointer; width: 100%;">Clear Canvas</button>
</div>

<script>
    (function() {
        var canvas = document.getElementById('sd_canvas');
        var ctx = canvas.getContext('2d');
        ctx.strokeStyle='white'; ctx.lineWidth=18; ctx.lineCap='round';
        var drawing = false;

        canvas.onmousedown=()=>drawing=true; 
        canvas.onmouseup=()=>{drawing=false; ctx.beginPath();};
        canvas.onmousemove=(e)=>{
            if(!drawing)return; 
            var r=canvas.getBoundingClientRect(); 
            ctx.lineTo(e.clientX-r.left, e.clientY-r.top); 
            ctx.stroke(); ctx.beginPath(); ctx.moveTo(e.clientX-r.left, e.clientY-r.top);
        };

        document.getElementById('sd_clear').onclick = () => ctx.clearRect(0, 0, 280, 280);

        document.getElementById('sd_predict').onclick = () => {
            var dataURL = canvas.toDataURL('image/png');
            // Find specifically the current showdown bridge
            var inputs = document.querySelectorAll('.sd-bridge-input input');
            var bridge = inputs[inputs.length - 1]; 
            bridge.value = dataURL;
            bridge.dispatchEvent(new Event('input', { bubbles: true }));
        };
    })();
</script>
"""

showdown_out = widgets.Output()
bridge_box = widgets.Box([showdown_bridge], layout=widgets.Layout(display='none'))

display(HTML(showdown_html))
display(bridge_box, showdown_out)

Box(children=(Text(value='', _dom_classes=('sd-bridge-input',)),), layout=Layout(display='none'))

Output()